# Standard A2C Training (GPU-Optimized `@tf.function`) — Seed 512

- **Algorithm:** Conventional synchronous A2C (`-mean(log(π(a|s)) * stop_gradient(δ)) - β * H(π)` from `training.py:420`).
- **Seed:** `512`
- **GPU Optimizations (100% mathematically identical to `A2C_512.ipynb`, ~10–30× faster on A100/T4):**
  1. Pure-TensorFlow `tf_quantile(x, q)` on GPU (eliminates ~900 GPU→CPU `.numpy()` syncs per episode).
  2. Preloads `train.tfrecords` once into GPU/RAM (`~0.8 MB`) instead of re-reading Google Drive 600 times.
  3. Compiles each 32-step window update with `@tf.function` (zero retracing across tail windows).
- **Outputs:** `./checkpoints/` (every 10 episodes, auto-resume) and `./logs/` (including `eval_metrics_seed512.json`).


In [ ]:
import os, sys, json, csv, time, random, pathlib
from datetime import datetime
from collections import defaultdict

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf

# Configure GPU memory growth (A100 / T4 / local GPU)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass
print(f"TensorFlow: {tf.__version__} | GPUs detected: {len(gpus)} {gpus}")

ALGORITHM = "A2C"
SEED = 512

# === Google Colab / Google Drive Paths ===
from google.colab import drive
drive.mount('/content/drive')

base_drive_path = '/content/drive/MyDrive/A2C_512/'
SEED_DIR = pathlib.Path(base_drive_path)
CHECKPOINT_DIR = SEED_DIR / "checkpoints"
LOG_DIR = SEED_DIR / "logs"
DATA_DIR = SEED_DIR / "data"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# === Local Paths (commented out for Colab) ===
# CWD = pathlib.Path.cwd()
# REPO_ROOT = CWD
# for cand in [CWD, CWD.parent, CWD.parent.parent, CWD.parent.parent.parent,
#              pathlib.Path(r"C:\GitHub\Q-learning-for-Inventory-Management")]:
#     if (cand / "data" / "train.tfrecords").exists():
#         REPO_ROOT = cand
#         break
# SEED_DIR = REPO_ROOT / "Feedback 23-11" / "task2" / f"A2C_{SEED}"
# CHECKPOINT_DIR = SEED_DIR / "checkpoints"
# LOG_DIR = SEED_DIR / "logs"
# DATA_DIR = REPO_ROOT / "data"
# CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
# LOG_DIR.mkdir(parents=True, exist_ok=True)

class FLAGS:
    seed = SEED
    algorithm = ALGORITHM
    train_file = str(DATA_DIR / "train.tfrecords")
    capacity_file = str(DATA_DIR / "capacity.tfrecords")
    stock_file = str(DATA_DIR / "stock.tfrecords")
    predict_file = str(DATA_DIR / "test.tfrecords")
    checkpoint_dir = str(CHECKPOINT_DIR)
    log_dir = str(LOG_DIR)

    # Identical hyperparameters to A2C-mod-seed.ipynb
    dropout_prob = 0.1
    train_episodes = 600
    num_products = 220
    num_timesteps = 900
    num_features = 3
    num_actions = 14
    hidden_size = 32
    entropy_coefficient = 0.001
    gamma = 0.99
    waste = 0.025
    actor_learning_rate = 0.001
    critic_learning_rate = 0.001
    zero_inventory = 1e-5
    batch_size = 32

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"OK: Set all seeds to {seed}")

set_seed(FLAGS.seed)
print(f"Data dir      : {DATA_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Log dir       : {LOG_DIR}")
assert pathlib.Path(FLAGS.train_file).exists(), f"Missing {FLAGS.train_file}"


## 1. Network Architecture, Pure-TF `tf_quantile`, & Logger


In [ ]:
class Dense(tf.Module):
    def __init__(self, input_dim, output_size, activation=None, stddev=1.0, seed=None):
        super().__init__()
        if seed is not None:
            initializer = tf.keras.initializers.TruncatedNormal(stddev=stddev, seed=seed)
            self.w = tf.Variable(initializer([input_dim, output_size]), name="w")
        else:
            self.w = tf.Variable(tf.random.truncated_normal([input_dim, output_size], stddev=stddev), name="w")
        self.b = tf.Variable(tf.zeros([output_size]), name="b")
        self.activation = activation

    def __call__(self, x):
        y = tf.matmul(x, self.w) + self.b
        if self.activation:
            y = self.activation(y)
        return y

class Actor(tf.Module):
    def __init__(self, num_features, num_actions, hidden_size, activation=tf.nn.relu, dropout_prob=0.1, seed=None):
        super().__init__()
        self.layer1 = Dense(num_features, hidden_size, activation=None, seed=seed)
        self.layer2 = Dense(hidden_size, hidden_size, activation=None, seed=seed)
        self.layer3 = Dense(hidden_size, hidden_size, activation=None, seed=seed)
        self.layer4 = Dense(hidden_size, num_actions, activation=None, seed=seed)
        self.activation = activation
        self.dropout_prob = dropout_prob

    def __call__(self, state, training=True):
        rate = self.dropout_prob if training else 0.0
        x = self.activation(self.layer1(state))
        if rate > 0: x = tf.nn.dropout(x, rate)
        x = self.activation(self.layer2(x))
        if rate > 0: x = tf.nn.dropout(x, rate)
        x = self.activation(self.layer3(x))
        if rate > 0: x = tf.nn.dropout(x, rate)
        return tf.nn.softmax(self.layer4(x), axis=-1)

class Critic(tf.Module):
    def __init__(self, num_features, hidden_size, activation=tf.nn.relu, dropout_prob=0.1, seed=None):
        super().__init__()
        self.layer1 = Dense(num_features, hidden_size, activation=None, seed=seed)
        self.layer2 = Dense(hidden_size, 1, activation=None, seed=seed)
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.activation = activation
        self.dropout_prob = dropout_prob

    def __call__(self, state, training=True):
        rate = self.dropout_prob if training else 0.0
        x = self.layer1(state)
        x = self.norm1(x)
        x = self.activation(x)
        if rate > 0: x = tf.nn.dropout(x, rate)
        x = self.layer2(x)
        return tf.squeeze(x, axis=-1, name="factor_squeeze")

def sales_parser(s):
    return tf.io.parse_single_example(s, {"sales": tf.io.FixedLenFeature([FLAGS.num_products], tf.float32)})

def capacity_parser(s):
    return tf.io.parse_single_example(s, {"capacity": tf.io.FixedLenFeature([FLAGS.num_products], tf.float32)})

def stock_parser(s):
    return tf.io.parse_single_example(s, {"stock": tf.io.FixedLenFeature([FLAGS.num_products], tf.float32)})

def waste(x):
    return FLAGS.waste * x

def tf_quantile(x, q):
    """Exact pure-TensorFlow equivalent of np.quantile(x, q) (linear interpolation) on GPU."""
    x_sorted = tf.sort(x, direction="ASCENDING")
    n = tf.cast(tf.shape(x_sorted)[0], tf.float32)
    pos = tf.cast(q, tf.float32) * (n - 1.0)
    lo = tf.cast(tf.math.floor(pos), tf.int32)
    hi = tf.cast(tf.math.ceil(pos), tf.int32)
    frac = pos - tf.cast(lo, tf.float32)
    return x_sorted[lo] + frac * (x_sorted[hi] - x_sorted[lo])

def cross_entropy(p, q):
    return -tf.reduce_mean(tf.reduce_sum(p * tf.math.log(tf.math.maximum(1e-15, q)), axis=1))

class TrainingLogger:
    def __init__(self, log_dir):
        os.makedirs(log_dir, exist_ok=True)
        self.log_dir = log_dir
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.metrics = defaultdict(list)
        self.episode_logs = []

    def log_step(self, global_step, experience_step, rewards, stockouts, waste_val, delta, critic_loss, entropy_adj, actor_loss):
        self.metrics["global_step"].append(int(global_step))
        self.metrics["experience_step"].append(int(experience_step))
        self.metrics["rewards"].append(float(rewards))
        self.metrics["stockouts"].append(float(stockouts))
        self.metrics["waste"].append(float(waste_val))
        self.metrics["delta"].append(float(delta))
        self.metrics["critic_loss"].append(float(critic_loss))
        self.metrics["entropy_adjusted"].append(float(entropy_adj))
        self.metrics["actor_loss"].append(float(actor_loss))

    def save_episode_logs(self, episode, write_per_episode_files=False):
        if not self.metrics or len(self.metrics["global_step"]) == 0:
            return
        if write_per_episode_files:
            json_file = os.path.join(self.log_dir, f"training_log_{self.timestamp}_episode_{episode:04d}.json")
            with open(json_file, "w") as f:
                json.dump(dict(self.metrics), f, indent=2)
        self.episode_logs.append({
            "episode": episode,
            "steps": len(self.metrics["global_step"]),
            "rewards_mean": float(np.mean(self.metrics["rewards"])),
            "rewards_std": float(np.std(self.metrics["rewards"])),
            "stockouts_mean": float(np.mean(self.metrics["stockouts"])),
            "waste_mean": float(np.mean(self.metrics["waste"])),
            "delta_mean": float(np.mean(self.metrics["delta"])),
            "critic_loss_mean": float(np.mean(self.metrics["critic_loss"])),
            "entropy_adjusted_mean": float(np.mean(self.metrics["entropy_adjusted"])),
            "actor_loss_mean": float(np.mean(self.metrics["actor_loss"])),
        })
        self.metrics = defaultdict(list)

    def save_summary(self):
        summary_file = os.path.join(self.log_dir, f"training_summary_{self.timestamp}.json")
        with open(summary_file, "w") as f:
            json.dump(self.episode_logs, f, indent=2)
        print(f"OK: Training summary saved to {summary_file}")
        return summary_file

# Verify tf_quantile matches np.quantile to < 1e-6
_test_x = np.random.rand(220).astype(np.float32)
assert abs(float(tf_quantile(tf.constant(_test_x), 0.95)) - float(np.quantile(_test_x, 0.95))) < 1e-6
assert abs(float(tf_quantile(tf.constant(_test_x), 0.05)) - float(np.quantile(_test_x, 0.05))) < 1e-6
print("OK: Networks, tf_quantile (verified exact), and TrainingLogger ready.")


## 2. GPU-Compiled `@tf.function` Training Loop (600 Episodes, Auto-Resume)


In [ ]:
ACTION_SPACE_CONST = tf.tile(
    tf.constant([[0.0, 0.005, 0.01, 0.0125, 0.015, 0.0175, 0.02, 0.03, 0.04, 0.08, 0.12, 0.2, 0.5, 1.0]], dtype=tf.float32),
    [FLAGS.num_products, 1],
)

def train_gpu():
    set_seed(FLAGS.seed)
    logger = TrainingLogger(log_dir=FLAGS.log_dir)

    # Preload capacity and entire normalized training sales ONCE into memory/GPU (~0.8 MB)
    capacity = next(iter(tf.data.TFRecordDataset(FLAGS.capacity_file).map(capacity_parser)))["capacity"]
    train_sales_raw = tf.stack(
        [r["sales"] for r in tf.data.TFRecordDataset(FLAGS.train_file).map(sales_parser)],
        axis=0,
    )
    train_sales_norm = tf.divide(train_sales_raw, capacity)  # [T_train, 220]
    total_T = int(train_sales_norm.shape[0])
    print(f"Preloaded train_sales_norm: {train_sales_norm.shape} (pre-sliced into sliding windows of {FLAGS.batch_size}, shift {FLAGS.batch_size - 1})")

    # Build exact same sliding windows as .window(batch_size, shift=batch_size-1, drop_remainder=False)
    windows = []
    start = 0
    step_shift = FLAGS.batch_size - 1
    while start < total_T:
        w = train_sales_norm[start : start + FLAGS.batch_size]
        if w.shape[0] >= 2:  # need at least 1 transition (s -> s')
            windows.append(w)
        start += step_shift

    actor_optimizer = tf.optimizers.Adam(FLAGS.actor_learning_rate)
    critic_optimizer = tf.optimizers.Adam(FLAGS.critic_learning_rate)

    actor = Actor(FLAGS.num_features, FLAGS.num_actions, FLAGS.hidden_size,
                  activation=tf.nn.relu, dropout_prob=FLAGS.dropout_prob, seed=FLAGS.seed)
    critic = Critic(FLAGS.num_features, FLAGS.hidden_size,
                    activation=tf.nn.relu, dropout_prob=FLAGS.dropout_prob, seed=FLAGS.seed)

    # Build variables once before @tf.function tracing
    _ = actor(tf.zeros([FLAGS.num_products, FLAGS.num_features], dtype=tf.float32), training=True)
    _ = critic(tf.zeros([FLAGS.num_products, FLAGS.num_features], dtype=tf.float32), training=True)

    global_step = tf.Variable(0)
    completed_episodes = tf.Variable(0)

    checkpoint_prefix = os.path.join(FLAGS.checkpoint_dir, "ckpt")
    checkpoint = tf.train.Checkpoint(
        critic_optimizer=critic_optimizer,
        actor_optimizer=actor_optimizer,
        critic=critic,
        actor=actor,
        step=global_step,
        completed_episodes=completed_episodes,
    )
    latest = tf.train.latest_checkpoint(FLAGS.checkpoint_dir)
    if latest is not None:
        checkpoint.restore(latest).expect_partial()
        print(f"[RESUME] Restored {latest} (completed_episodes={int(completed_episodes)}, global_step={int(global_step)})")
    else:
        print(f"[START]  Fresh GPU Standard A2C training | Seed={FLAGS.seed} | Episodes={FLAGS.train_episodes}")

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[FLAGS.num_products], dtype=tf.float32),
        tf.TensorSpec(shape=[None, FLAGS.num_products], dtype=tf.float32),
    ])
    def train_window_step(x_start, sales_window):
        n_trans = tf.shape(sales_window)[0] - 1
        with tf.GradientTape() as actor_tape, tf.GradientTape() as critic_tape:
            exp_s = tf.TensorArray(size=n_trans, dtype=tf.float32, element_shape=[FLAGS.num_products, FLAGS.num_features])
            exp_p = tf.TensorArray(size=n_trans, dtype=tf.float32, element_shape=[FLAGS.num_products, FLAGS.num_actions])
            exp_pu = tf.TensorArray(size=n_trans, dtype=tf.float32, element_shape=[FLAGS.num_products])
            exp_s_prime = tf.TensorArray(size=n_trans, dtype=tf.float32, element_shape=[FLAGS.num_products, FLAGS.num_features])
            exp_r = tf.TensorArray(size=n_trans, dtype=tf.float32, element_shape=[FLAGS.num_products])
            exp_z = tf.TensorArray(size=n_trans, dtype=tf.float32, element_shape=[FLAGS.num_products])
            exp_q = tf.TensorArray(size=n_trans, dtype=tf.float32, element_shape=[FLAGS.num_products])

            x = x_start
            q = waste(x)
            sales = sales_window[0]
            s = tf.transpose(tf.stack([x, sales, q], axis=0), perm=[1, 0])
            policy_probs = actor(s, training=True)

            def cond(t, *_):
                return t < n_trans

            def body(t, x, q, s, sales, policy_probs, exp_s, exp_p, exp_pu, exp_s_prime, exp_r, exp_z, exp_q):
                sales_prime = sales_window[t + 1]
                policy_index = tf.squeeze(tf.random.categorical(tf.math.log(tf.math.maximum(1e-15, policy_probs)), 1), axis=-1)
                policy_mask = tf.one_hot(policy_index, FLAGS.num_actions)
                policy_selected = tf.boolean_mask(policy_probs, policy_mask)
                u = tf.boolean_mask(ACTION_SPACE_CONST, policy_mask)

                overstock = tf.math.maximum(0.0, (x + u) - 1.0)
                x_u = tf.math.minimum(1.0, x + u)
                x_prime = tf.math.maximum(0.0, x_u - sales)

                q_prime = waste(x_prime)
                s_prime = tf.transpose(tf.stack([x_prime, sales_prime, q_prime], axis=0), perm=[1, 0])

                z = tf.cast(x < FLAGS.zero_inventory, tf.float32)
                quan_val = tf_quantile(x, 0.95) - tf_quantile(x, 0.05)
                quan = tf.fill([FLAGS.num_products], quan_val)
                r = 1.0 - z - overstock - q - quan

                exp_s = exp_s.write(t, s)
                exp_p = exp_p.write(t, policy_probs)
                exp_pu = exp_pu.write(t, policy_selected)
                exp_s_prime = exp_s_prime.write(t, s_prime)
                exp_r = exp_r.write(t, r)
                exp_z = exp_z.write(t, z)
                exp_q = exp_q.write(t, q)

                next_probs = actor(s_prime, training=True)
                return (t + 1, x_prime, q_prime, s_prime, sales_prime, next_probs,
                        exp_s, exp_p, exp_pu, exp_s_prime, exp_r, exp_z, exp_q)

            (_, x, q, s, sales, policy_probs,
             exp_s, exp_p, exp_pu, exp_s_prime, exp_r, exp_z, exp_q) = tf.while_loop(
                cond, body,
                (tf.constant(0), x, q, s, sales, policy_probs,
                 exp_s, exp_p, exp_pu, exp_s_prime, exp_r, exp_z, exp_q),
            )

            s_batch = tf.reshape(exp_s.stack(), [-1, FLAGS.num_features])
            p_batch = tf.reshape(exp_p.stack(), [-1, FLAGS.num_actions])
            pu_batch = tf.reshape(exp_pu.stack(), [-1])
            s_prime_batch = tf.reshape(exp_s_prime.stack(), [-1, FLAGS.num_features])
            r_batch = tf.reshape(exp_r.stack(), [-1])
            z_batch = tf.reshape(exp_z.stack(), [-1])
            q_batch = tf.reshape(exp_q.stack(), [-1])

            v = critic(s_batch, training=True)
            v_prime = critic(s_prime_batch, training=True)
            delta = (r_batch + FLAGS.gamma * v_prime) - v

            critic_loss = 0.5 * tf.reduce_mean(tf.math.square(delta))
            entropy_p = cross_entropy(p_batch, p_batch)
            entropy_adj = FLAGS.entropy_coefficient * entropy_p

            # Standard A2C Actor Loss (training.py:420)
            actor_loss = -tf.reduce_mean(
                tf.math.log(tf.math.maximum(1e-15, pu_batch)) * tf.stop_gradient(delta)
            ) - entropy_adj

        actor_grads = actor_tape.gradient(actor_loss, actor.variables)
        critic_grads = critic_tape.gradient(critic_loss, critic.variables)
        actor_optimizer.apply_gradients(zip(actor_grads, actor.variables))
        critic_optimizer.apply_gradients(zip(critic_grads, critic.variables))
        global_step.assign_add(1)

        return (
            x,
            tf.reduce_mean(r_batch),
            tf.reduce_mean(z_batch),
            tf.reduce_mean(q_batch),
            tf.reduce_mean(delta),
            critic_loss,
            entropy_adj,
            actor_loss,
            n_trans,
        )

    start_ep = int(completed_episodes)
    t_start = time.time()

    for episode in range(start_ep, FLAGS.train_episodes):
        ep_t0 = time.time()
        x = tf.random.uniform(shape=[FLAGS.num_products], minval=0.0, maxval=1.0, dtype=tf.float32)

        for w in windows:
            x, r_m, z_m, q_m, d_m, c_l, e_a, a_l, n_tr = train_window_step(x, w)
            logger.log_step(int(global_step), int(n_tr), float(r_m), float(z_m), float(q_m), float(d_m), float(c_l), float(e_a), float(a_l))

        completed_episodes.assign(episode + 1)
        # Save compact summary in memory every episode; write checkpoint & print every 10 episodes (avoids 1,200 Drive file writes!)
        logger.save_episode_logs(episode + 1, write_per_episode_files=False)

        if (episode + 1) % 10 == 0:
            ckpt_path = checkpoint.save(file_prefix=checkpoint_prefix)
            logger.save_summary()
            ep_sum = logger.episode_logs[-1]
            print(
                f"[A2C-GPU Seed {FLAGS.seed}] Ep {episode + 1:03d}/{FLAGS.train_episodes} | "
                f"reward={ep_sum['rewards_mean']:.4f} | stockout={ep_sum['stockouts_mean']:.4f} | "
                f"waste={ep_sum['waste_mean']:.4f} | H*beta={ep_sum['entropy_adjusted_mean']:.5f} | "
                f"ep_time={time.time()-ep_t0:.2f}s | saved {os.path.basename(ckpt_path)}"
            )

    logger.save_summary()
    print(f"DONE: Seed {FLAGS.seed} finished in {(time.time() - t_start)/60:.2f} min.")
    return actor

actor_trained = train_gpu()


## 3. Held-Out Test Evaluation (ALL / EASY / MEDIUM / HARD for Table 6 & Table 7)


In [ ]:
def evaluate_a2c_on_test(actor_model):
    capacity = next(iter(tf.data.TFRecordDataset(FLAGS.capacity_file).map(capacity_parser)))["capacity"].numpy()
    x_init = next(iter(tf.data.TFRecordDataset(FLAGS.stock_file).map(stock_parser)))["stock"].numpy()
    test_sales_raw = np.array(
        [r["sales"].numpy() for r in tf.data.TFRecordDataset(FLAGS.predict_file).map(sales_parser)],
        dtype=np.float32,
    )
    test_sales = test_sales_raw / capacity
    action_levels = np.array([0.0, 0.005, 0.01, 0.0125, 0.015, 0.0175, 0.02, 0.03, 0.04, 0.08, 0.12, 0.2, 0.5, 1.0], dtype=np.float32)

    scenarios = {
        "ALL":    {"x_scale": 1.0, "sales_scale": 1.0, "waste_rate": 0.025},
        "EASY":   {"x_scale": 0.3, "sales_scale": 0.2, "waste_rate": 0.010},
        "MEDIUM": {"x_scale": 0.6, "sales_scale": 0.5, "waste_rate": 0.050},
        "HARD":   {"x_scale": 0.9, "sales_scale": 0.8, "waste_rate": 0.150},
    }

    results = {}
    for sc_name, cfg in scenarios.items():
        x = np.clip(x_init * cfg["x_scale"], 0.0, 1.0).astype(np.float32)
        rewards_list, srv_list, hold_list, wst_list, ord_list, so_list = [], [], [], [], [], []
        for t in range(len(test_sales)):
            sales_t = (test_sales[t] * cfg["sales_scale"]).astype(np.float32)
            q_t = (x * cfg["waste_rate"]).astype(np.float32)
            s_t = np.stack([x, sales_t, q_t], axis=1)
            probs = actor_model(tf.constant(s_t, dtype=tf.float32), training=False).numpy()
            actions = np.argmax(probs, axis=-1)
            u = action_levels[actions]

            overstock = np.maximum(0.0, (x + u) - 1.0)
            x_u = np.minimum(1.0, x + u)
            z = (x < FLAGS.zero_inventory).astype(np.float32)
            quan = float(np.quantile(x, 0.95) - np.quantile(x, 0.05))
            r = 1.0 - z - overstock - q_t - quan

            rewards_list.append(float(r.mean()))
            srv_list.append(float((1.0 - z).mean()))
            hold_list.append(float(overstock.mean()))
            wst_list.append(float(q_t.mean()))
            ord_list.append(quan)
            so_list.append(float(z.mean()))

            x = np.maximum(0.0, x_u - sales_t).astype(np.float32)

        results[sc_name] = {
            "total_reward": float(np.mean(rewards_list)),
            "service_level": float(np.mean(srv_list)),
            "holding_cost": float(np.mean(hold_list)),
            "waste_cost": float(np.mean(wst_list)),
            "ordering_cost": float(np.mean(ord_list)),
            "stockout_rate": float(np.mean(so_list)),
        }

    out_json = os.path.join(FLAGS.log_dir, f"eval_metrics_seed{FLAGS.seed}.json")
    with open(out_json, "w") as f:
        json.dump({"algorithm": "A2C", "seed": FLAGS.seed, "metrics": results}, f, indent=2)
    print(f"OK: Saved evaluation metrics to {out_json}")
    for sc, m in results.items():
        print(f"  {sc:6s} | Reward={m['total_reward']:+.4f} | Srv={m['service_level']:.4f} | Hold={m['holding_cost']:.4f} | Waste={m['waste_cost']:.4f} | Ord={m['ordering_cost']:.4f} | SO={m['stockout_rate']:.4f}")
    return results

eval_results = evaluate_a2c_on_test(actor_trained)
